# Context-Deference — Phase 2: causal steering & subspace removal

Turns the correlational cosine result into a **causal** one. Three checks:

1. **Necessity** — ablating a behavior's suppression direction should *reduce* its override rate.
2. **Random-direction baseline** — a random matched-norm direction should *not* reproduce that
   effect (else the direction isn't special).
3. **Subspace-removal consistency** (arXiv:2509.21305) — remove behavior *B*'s suppression subspace
   and measure every behavior. Diagonal collapses + off-diagonal stable ⇒ **distinct/separable**
   suppressors; everything collapsing together ⇒ a **shared** suppressor.

Needs a GPU. Reuses `src/pipeline.py` so extraction matches the MVP driver exactly.

In [1]:
# On Colab: !pip install -q -r ../requirements.txt
import sys, os
sys.path.append(os.path.abspath(".."))
import numpy as np, torch
from src import model as M, data as D, pipeline as PL, steering as St, eval as Ev

torch.set_grad_enabled(False)
# Env overrides (same as the driver): defaults = science run; for a local check e.g.
#   CD_MODEL=qwen2.5-1.5b-instruct CD_MAX_PAIRS=16 CD_MAX_NEW_TOKENS=24 CD_N_RANDOM=4 CD_CONTRAST=manip_vs_clean
cfg        = D.load_behaviors_config("../configs/behaviors.yaml")
models_cfg = D.load_yaml("../configs/models.yaml")
mc = models_cfg["models"][os.environ.get("CD_MODEL", models_cfg["default_model"])]
LAYER, POS   = mc["default_layer"], mc["default_position"]
CONTRAST     = os.environ.get("CD_CONTRAST", cfg["contrast"]["mode"])   # CD_CONTRAST to A/B a contrast
SUBSET       = os.environ["CD_SUBSET"].split(",") if os.environ.get("CD_SUBSET") else cfg["mvp_subset"]
N_EVAL       = int(os.environ.get("CD_MAX_PAIRS", "48"))     # prompts/behavior under intervention
MAX_NEW_TOKENS = int(os.environ.get("CD_MAX_NEW_TOKENS", "64"))
N_RANDOM     = int(os.environ.get("CD_N_RANDOM", "8"))
print("model:", mc["tl_name"], "| contrast:", CONTRAST, "| behaviors:", SUBSET, "| N_EVAL:", N_EVAL, "| N_RANDOM:", N_RANDOM)

/Users/childrebelsoldier/suppression/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


model: Qwen/Qwen2.5-1.5B-Instruct | contrast: manip_vs_clean | behaviors: ['safety', 'knowledge_conflict'] | N_EVAL: 6 | N_RANDOM: 2


In [2]:
bundle = M.load_model(mc["tl_name"], dtype=mc.get("dtype", "bfloat16"))
print("loaded", bundle.name, "| layers:", bundle.n_layers, "| d_model:", bundle.d_model)

/Users/childrebelsoldier/suppression/.venv/lib/python3.11/site-packages/transformer_lens/config/hooked_transformer_config.py:354: UserWarning: MPS backend may produce silently incorrect results (PyTorch 2.12.1). Set TRANSFORMERLENS_ALLOW_MPS=1 to suppress this warning. See: https://github.com/TransformerLensOrg/TransformerLens/issues/1178
  warn_if_mps(self.device)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/338 [00:03<21:00,  3.74s/it]

Loading weights:  51%|█████     | 171/338 [00:03<00:02, 61.00it/s]

Loading weights:  67%|██████▋   | 227/338 [00:04<00:01, 62.95it/s]

...

Loading weights:  96%|█████████▌| 323/338 [00:06<00:00, 58.57it/s]

Loading weights:  99%|█████████▊| 333/338 [00:06<00:00, 55.31it/s]

Loading weights: 100%|██████████| 338/338 [00:06<00:00, 49.18it/s]

Loaded pretrained model Qwen/Qwen2.5-1.5B-Instruct into HookedTransformer


loaded Qwen/Qwen2.5-1.5B-Instruct | layers: 28 | d_model: 1536


In [3]:
# --- extract suppression directions + manip prompts (one GPU pass/behavior, shared pipeline) ---
sup_dirs, prompts_by, pairs_by = {}, {}, {}
for b in SUBSET:
    print("extracting:", b)
    ex = PL.extract_behavior(bundle, b, cfg, layer=LAYER, position=POS, contrast_mode=CONTRAST,
                             max_items=N_EVAL, max_new_tokens=MAX_NEW_TOKENS)
    sup_dirs[b]   = ex.suppression_dir
    prompts_by[b] = ex.prompts["manip"]
    pairs_by[b]   = ex.pairs
    M.free()
# 1-D subspace to remove per behavior = its suppression direction (widen with residual/PCs if desired)
subspace_by = {b: sup_dirs[b].vec[None, :] for b in SUBSET}

extracting: safety


extracting: knowledge_conflict


In [4]:
# --- (1) necessity + (2) random-direction baseline ---
# ablate the suppression direction across all layers; the override rate should fall below baseline
# AND below what random matched-norm directions produce.
for b in SUBSET:
    res = St.random_direction_baseline(bundle, b, sup_dirs[b], prompts_by[b], pairs_by[b],
                                       n_random=N_RANDOM, seed=0, max_new_tokens=MAX_NEW_TOKENS)
    p = Ev.random_baseline_pvalue(res["real_ablated"], res["random_ablated"], "decrease")
    print(f"{b:<20} baseline={res['baseline']:.2f}  real_ablated={res['real_ablated']:.2f}  "
          f"random_mean={np.mean(res['random_ablated']):.2f}  p={p:.3f}")
    M.free()

safety               baseline=1.00  real_ablated=1.00  random_mean=1.00  p=1.000


knowledge_conflict   baseline=0.50  real_ablated=0.50  random_mean=0.50  p=1.000


In [5]:
# --- (3) subspace-removal consistency matrix ---
cc = St.subspace_removal_consistency_check(bundle, SUBSET, subspace_by, prompts_by, pairs_by,
                                           max_new_tokens=MAX_NEW_TOKENS)
print("baseline override rates:", {k: round(v, 2) for k, v in cc["baseline"].items()})
print("\nrows = removed subspace, cols = measured behavior's override rate")
print(" " * 20 + "".join(f"{b[:12]:>13}" for b in SUBSET))
for rem in SUBSET:
    print(f"{rem:<20}" + "".join(f"{cc['matrix'][rem][meas]:>13.2f}" for meas in SUBSET))
print("\nsummary (own_drop >> other_drop => selective/distinct):")
for b, s in cc["summary"].items():
    print(f"  {b:<20} own_drop={s['own_drop']:+.2f}  other_drop={s['other_drop']:+.2f}  selective={s['selective']}")

baseline override rates: {'safety': 1.0, 'knowledge_conflict': 0.5}

rows = removed subspace, cols = measured behavior's override rate
                           safety knowledge_co
safety                       1.00         1.00
knowledge_conflict           1.00         0.50

summary (own_drop >> other_drop => selective/distinct):
  safety               own_drop=+0.00  other_drop=-0.50  selective=True
  knowledge_conflict   own_drop=+0.00  other_drop=+0.00  selective=False


## Reading the causal result

- **Necessity + baseline:** `real_ablated` well below `baseline` and below `random_mean` (small `p`)
  ⇒ that suppression direction is *causally* responsible for the override — not any direction will do.
- **Consistency matrix:** the diagonal is the override rate of a behavior after removing *its own*
  suppression subspace; off-diagonal is a behavior's rate after removing *another's*.
  - Diagonal collapses, off-diagonal ≈ baseline ⇒ **distinct/separable** suppressors (`selective=True`).
  - Removing any one subspace collapses all ⇒ a **shared** suppressor.
  - Partial cross-effects ⇒ **distinct-but-shared-knob** — corroborate with the residual-cosine PCA
    (`subspace.shared_knob_test`) and principal angles from the MVP driver.

Together with the correlational matrices this is the full pre-registered read: geometry (cosines/PCA)
+ causality (ablation necessity, random baseline, subspace-removal). A null (distinct) remains a result.